# Load Dataset

In [1]:
from beir.datasets.data_loader import GenericDataLoader

data_path = "scifact"

corpus, queries, qrels = GenericDataLoader(
    data_path
).load(split="test")

C:\Users\lenovo\nlp_env\Lib\site-packages\beir\datasets\data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
100%|██████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 110273.40it/s]


In [2]:
import pandas as pd

corpus_df = pd.DataFrame.from_dict(
    corpus,
    orient="index" 
).reset_index()

In [3]:
pd.set_option('display.max_colwidth', None)

In [4]:
corpus_df['documents'] = [
    
    row['title'] + ' ' + row['text'] if row['title'].endswith('.') else row['title'] + ". " + row['text']
    for _, row in corpus_df.iterrows()
]

# Preprocessing & Tokenization

In [5]:
# we will just remove punctuations and apply lower()
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

nltk.download("punkt")
nltk.download("punkt_tab")

def preprocess(text):
    tokens = word_tokenize(text.lower())
    return [
        token for token in tokens
        if token not in string.punctuation
        and token not in {"+", "=", "-", "/", "*"}
    ]

corpus_df["docs"] = corpus_df["documents"].apply(preprocess)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [7]:
corpus_df

,index,text,title,documents,docs
0,4983,"Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at both times were similar (1.2 versus 1.1 microm2/ms). Relative anisotropy was higher the closer birth was to term with greater absolute values in the internal capsule than in the central white matter. Preterm infants at term showed higher mean diffusion coefficients in the central white matter (1.4 +/- 0.24 versus 1.15 +/- 0.09 microm2/ms, p = 0.016) and lower relative anisotropy in both areas compared with full-term infants (white matter, 10.9 +/- 0.6 versus 22.9 +/- 3.0%, p = 0.001; internal capsule, 24.0 +/- 4.44 versus 33.1 +/- 0.6% p = 0.006). Nonmyelinated fibers in the corpus callosum were visible by diffusion tensor MRI as early as 28 wk; full-term and preterm infants at term showed marked differences in white matter fiber organization. The data indicate that quantitative assessment of water diffusion by diffusion tensor MRI provides insight into microstructural development in cerebral white matter in living infants.",Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.,"Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at both times were similar (1.2 versus 1.1 microm2/ms). Relative anisotropy was higher the closer birth was to term with greater absolute values in the internal capsule than in the central white matter. Preterm infants at term showed higher mean diffusion coefficients in the central white matter (1.4 +/- 0.24 versus 1.15 +/- 0.09 microm2/ms, p = 0.016) and lower relative anisotropy in both areas compared with full-term infants (white matter, 10.9 +/- 0.6 versus 22.9 +/- 3.0%, p = 0.001; internal capsule, 24.0 +/- 4.44 versus 33.1 +/- 0.6% p = 0.006). Nonmyelinated fibers in the corpus callosum were visible by diffusion tensor MRI as early as 28 wk; full-term and preterm infants at term showed marked differences in white matter fiber organization. The data indicate that quantitative assessment of water diffusion by diffusion tensor MRI provides insight into microstructural development in cerebral white matter in living infants.","[microstructural, development, of, human, newborn, cerebral, white, mat

# BM25 Indexing

In [8]:
from rank_bm25 import BM25Okapi

tokenized_corpus = corpus_df['docs'].tolist()
bm25 = BM25Okapi(tokenized_corpus)

In [9]:
# Load Queries & Qrels

In [10]:
queries_df = (pd.DataFrame.from_dict(queries, orient="index")
              .reset_index()
              .rename(columns={'index': 'query_id', 0: 'query_text'}))

queries_df

,query_id,query_text
0,1,0-dimensional biomaterials show inductive properties.
1,3,"1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth weight.
4,36,A deficiency of vitamin B12 increases blood levels of homocysteine.
...,...,...
295,1379,Women with a higher birth weight are more likely to develop breast cancer later in life.
296,1382,aPKCz causes tumour enhancement by affecting glutamine metabolism.
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels through xCT inhibition.


In [11]:
qrels

{'1': {'31715818': 1},
 '3': {'14717500': 1},
 '5': {'13734012': 1},
 '13': {'1606628': 1},
 '36': {'5152028': 1, '11705328': 1},
 '42': {'18174210': 1},
 '48': {'13734012': 1},
 '49': {'5953485': 1},
 '50': {'12580014': 1},
 '51': {'45638119': 1},
 '53': {'45638119': 1},
 '54': {'49556906': 1},
 '56': {'4709641': 1},
 '57': {'4709641': 1},
 '70': {'5956380': 1, '4414547': 1},
 '72': {'6076903': 1},
 '75': {'4387784': 1},
 '94': {'1215116': 1},
 '99': {'18810195': 1},
 '100': {'4381486': 1},
 '113': {'6157837': 1},
 '115': {'33872649': 1},
 '118': {'6372244': 1},
 '124': {'4883040': 1},
 '127': {'21598000': 1},
 '128': {'8290953': 1},
 '129': {'27768226': 1},
 '130': {'27768226': 1},
 '132': {'7975937': 1},
 '133': {'38485364': 1,
  '6969753': 1,
  '17934082': 1,
  '16280642': 1,
  '12640810': 1},
 '137': {'26016929': 1},
 '141': {'6955746': 1, '14437255': 1},
 '142': {'10582939': 1},
 '143': {'10582939': 1},
 '146': {'10582939': 1},
 '148': {'1084345': 1},
 '163': {'18872233': 1},
 '1

# Evaluation - Original Queries

In [12]:
qrels_list = []
for query_id, docs in qrels.items():
    for doc_id, relevance in docs.items():
        qrels_list.append({
            'query_id': query_id,
            'doc_id': doc_id,
            'relevance': relevance
        })

qrels_df = pd.DataFrame(qrels_list)
qrels_df.head(10)

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
5,36,11705328,1
6,42,18174210,1
7,48,13734012,1
8,49,5953485,1
9,50,12580014,1


In [13]:
import numpy as np

def cal_scores(queries):
    run_dict = {}

    for val, query in queries.items():
        scores = bm25.get_scores(preprocess(query))
        top_indices = np.argsort(-scores)[:10]
        top_doc_ids = corpus_df.iloc[top_indices]["index"].tolist()
        scores_dict = {}

        for doc_id, idx in zip(top_doc_ids, top_indices):
            scores_dict[doc_id] = float(scores[idx])

        run_dict[val] = scores_dict

    return run_dict

In [14]:
run_dict = cal_scores(queries)

In [15]:
run_dict

{'1': {'10931595': 8.944874242044964,
  '10906636': 8.095597499525981,
  '13231899': 7.449437666936065,
  '12824568': 7.437830035846066,
  '42240424': 7.323774541156759,
  '24998637': 6.877014179282162,
  '825728': 6.853055124546106,
  '18953920': 6.541494468113682,
  '4465608': 6.529638488722967,
  '17518195': 6.51738322715328},
 '3': {'3672261': 43.45984912975732,
  '2739854': 42.29905542175247,
  '14717500': 42.13393233504449,
  '23389795': 41.339045178987135,
  '4632921': 40.51069806110948,
  '4414547': 38.83058925686706,
  '19497526': 36.19331447382098,
  '13519661': 35.88909733363362,
  '14019636': 34.07899909884748,
  '15155862': 32.68735851042899},
 '5': {'13734012': 24.680813763745853,
  '18617259': 19.36676006154603,
  '17333231': 17.419534650181035,
  '42240424': 17.018608143500344,
  '695938': 15.656779866481456,
  '1292369': 14.763874754605506,
  '1171121': 12.894738851717513,
  '103007': 12.827914259897257,
  '16737163': 12.818726551267098,
  '11880289': 12.17851275710790

In [16]:
from ranx import Qrels, Run, evaluate

results = evaluate(
    qrels,
    run_dict,
    [
        "recall@5",
        "recall@10",
        "mrr@10",
        "ndcg@10"
    ]
)

print(results)

{'recall@5': np.float64(0.6952777777777777), 'recall@10': np.float64(0.7591666666666667), 'mrr@10': np.float64(0.5945343915343915), 'ndcg@10': np.float64(0.6306727337433231)}


In [17]:
import pandas as pd
from ranx import Qrels, Run, evaluate

# Create Qrels and Run objects
qrels_obj = qrels
run_obj = Run(run_dict)

# Get overall results
overall_results = evaluate(
    qrels_obj,
    run_obj,
    ["recall@5", "recall@10", "mrr@10", "ndcg@10"]
)

# Get per-query results by iterating over each query
per_query_results = {}

for query_id in qrels.keys():
    if query_id in run_dict:
        # Create single-query Qrels and Run
        single_qrels = Qrels({query_id: qrels[query_id]})
        single_run = Run({query_id: run_dict[query_id]})
        
        # Evaluate for this single query
        single_results = evaluate(
            single_qrels,
            single_run,
            ["recall@5", "recall@10", "mrr@10", "ndcg@10"]
        )
        
        per_query_results[query_id] = single_results

# Convert to DataFrame
df_per_query = pd.DataFrame(per_query_results).T
df_per_query.index.name = 'query_id'
df_per_query = df_per_query.reset_index()

In [19]:
df_per_query

,query_id,recall@5,recall@10,mrr@10,ndcg@10
0,1,0.00,0.0,0.000000,0.000000
1,3,1.00,1.0,0.333333,0.500000
2,5,1.00,1.0,1.000000,1.000000
3,13,0.00,0.0,0.000000,0.000000
4,36,0.00,0.5,0.166667,0.218407
...,...,...,...,...,...
295,1379,0.25,0.5,0.333333,0.334246
296,1382,0.00,0.0,0.000000,0.000000
297,1385,1.00,1.0,1.000000,1.000000
298,1389,1.00,1.0,1.000000,1.000000


# Expanded Queries

In [20]:
expanded_queries_df = pd.read_csv(
    "scifact_gemini_expanded_queries.csv",
    dtype={"query_id": str}
)

In [25]:
expanded_queries_df['expanded+original'] = expanded_queries_df['original_query'] + ' ' + expanded_queries_df['expanded_query']
expanded_queries_df

,query_id,original_query,expanded_query,expanded+original
0,1,0-dimensional biomaterials show inductive properties.,"zero-dimensional biomaterials, 0D biomaterials, 0D biocompatible materials, inductive capacity, bioinductive potential, inductive behavior, bioinduction capabilities","0-dimensional biomaterials show inductive properties. zero-dimensional biomaterials, 0D biomaterials, 0D biocompatible materials, inductive capacity, bioinductive potential, inductive behavior, bioinduction capabilities"
1,3,"1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.","1KGP, 1000 Genomes, genomic sequence variation, rare genetic variants, high-penetrance alleles, common genetic variants, sequence polymorphisms","1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants. 1KGP, 1000 Genomes, genomic sequence variation, rare genetic variants, high-penetrance alleles, common genetic variants, sequence polymorphisms"
2,5,1/2000 in UK have abnormal PrP positivity.,"abnormal prion protein prevalence United Kingdom, PrPSc carrier rate UK, prevalence of disease-associated prion protein, subclinical prion infection prevalence UK, abnormal PrP positivity rate Great Britain","1/2000 in UK have abnormal PrP positivity. abnormal prion protein prevalence United Kingdom, PrPSc carrier rate UK, prevalence of disease-associated prion protein, subclinical prion infection prevalence UK, abnormal PrP positivity rate Great Britain"
3,13,5% of perinatal mortality is due to low birth weight.,"perinatal death, perinatal demise, low birthweight, LBW, five percent, attributable to low birth weight, caused by low birth weight","5% of perinatal mortality is due to low birth weight. perinatal death, perinatal demise, low birthweight, LBW, five percent, attributable to low birth weight, caused by low birth weight"
4,36,A deficiency of vitamin B12 increases blood levels of homocysteine.,"cobalamin deficiency, hypocobalaminemia, hyperhomocysteinemia, elevated plasma homocysteine, elevated serum homocysteine, vitamin B-12 deficiency","A deficiency of vitamin B12 increases blood levels of homocysteine. cobalamin deficiency, hypocobalaminemia, hyperhomocysteinemia, elevated plasma homocysteine, elevated serum homocysteine, vitamin B-12 deficiency"
...,...,...,...,...
295,1379,Women with a higher birth weight are more likely to develop breast cancer later in life.,"neonatal weight, birth mass, female breast carcinoma, mammary gland cancer, birth size","Women with a higher birth weight are more likely to develop breast cancer later in life. neonatal weight, birth mass, female breast carcinoma, mammary gland cancer, birth size"
296,1382,aPKCz causes tumour enhancement by affecting glutamine metabolism.,"PRKCZ, atypical protein kinase C zeta, tumor progression, neoplastic growth, glutaminolysis","aPKCz causes tumour enhancement by affecting glutamine metabolism. PRKCZ, atypical protein kinase C zeta, tumor progression, neoplastic growth, glutaminolysis"
297,1385,cSMAC formation enhances weak ligand signalling.,"central supramolecular activation cluster, central supramolecular activation complex, immunological synapse, T-cell receptor signaling, TCR signaling","cSMAC formation enhances weak ligand signalling. central supramolecular activation cluster, central supramolecular activation complex, immunological synapse, T-cell receptor signaling, TCR signaling"
298,1389,mTORC2 regulates intracellular cysteine levels through xCT inhibition.,"mammalian target of rapamycin complex 2, mechanistic target of rapamycin complex 2, SLC7A11, cystine-glutamate antiporter, intracellular cysteine, amino acid metabolism","mTORC2 regulates intracellular cysteine levels through xCT inhibition. mammalian target of rapamycin complex 2, mechanistic target of rapamycin complex 2, SLC7A11, cystine-glutamate antiporter,

In [26]:
expanded_queries_dict = dict(
    zip(
        expanded_queries_df["query_id"],
        expanded_queries_df["expanded+original"]
    )
)

In [27]:
expanded_queries_dict

{'1': '0-dimensional biomaterials show inductive properties. zero-dimensional biomaterials, 0D biomaterials, 0D biocompatible materials, inductive capacity, bioinductive potential, inductive behavior, bioinduction capabilities',
 '3': '1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants. 1KGP, 1000 Genomes, genomic sequence variation, rare genetic variants, high-penetrance alleles, common genetic variants, sequence polymorphisms',
 '5': '1/2000 in UK have abnormal PrP positivity. abnormal prion protein prevalence United Kingdom, PrPSc carrier rate UK, prevalence of disease-associated prion protein, subclinical prion infection prevalence UK, abnormal PrP positivity rate Great Britain',
 '13': '5% of perinatal mortality is due to low birth weight. perinatal death, perinatal demise, low birthweight, LBW, five percent, attributable to low birth weight, caused by low birth weight',
 '36': 'A defi

In [28]:
expanded_run_dict = cal_scores(expanded_queries_dict)

# Evaluation - Expanded Queries

In [30]:
from ranx import Qrels, Run, evaluate

exp_results = evaluate(
    qrels,
    expanded_run_dict,
    [
        "recall@5",
        "recall@10",
        "mrr@10",
        "ndcg@10"
    ]
)
exp_results

{'recall@5': np.float64(0.7412222222222222),
 'recall@10': np.float64(0.8111666666666667),
 'mrr@10': np.float64(0.6343267195767195),
 'ndcg@10': np.float64(0.6724710569182905)}

In [31]:
results

{'recall@5': np.float64(0.6952777777777777),
 'recall@10': np.float64(0.7591666666666667),
 'mrr@10': np.float64(0.5945343915343915),
 'ndcg@10': np.float64(0.6306727337433231)}